In [4]:
import numpy as np
import scipy.special as sp

# 0) Game Definition & Evolutionary Dynamics

In this section we define the collective-risk social dilemma game in terms of the possible payoffs. After which the extensions discussed in the paper are implemented.

## 0.1) Collective-risk social dilemma game

In [5]:
class SocialDilemmaGame:
    def __init__(self, b, M, r, c):
        self.b = b
        self.M = M 
        self.r = r
        self.c = c
    # helper function - defines the Heaviside function
    def heaviside(self, x):
        if x < 0:
            return 0
        else:
            return 1

    # returns the payoff for both defectors and cooperators
    def payoff(self, j_c:int):
        h = self.heaviside(j_c - self.M)
        payoff_defectors = self.b * h + (1 - self.r) * self.b * (1 - h)
        payoff_cooperators = payoff_defectors - self.c
        return payoff_defectors, payoff_cooperators

## 0.2) Extension on the Collective-risk social dilemma game

In [ ]:
# fixed incentives
class FixedIncentives(SocialDilemmaGame):
    def __init__(self, b, M, r, c, alpha, pi_e, pi_t):
        super().__init__(b, M, r, c)
        self.alpha = alpha
        self.pi_e = pi_e
        self.pi_t = pi_t

    def payoff(self, j_c:int, j_e:int):
        p_d, _ = super().payoff(j_c + j_e) # super payoff
        payoff_for_defectors = p_d - (1 - self.alpha) * self.pi_e * self.delta
        payoff_for_cooperators = p_d + self.alpha * self.pi_e * self.delta - self.c
        payoff_for_executors = p_d + self.alpha * self.pi_e * self.delta - self.c - self.pi_t
        return payoff_for_defectors, payoff_for_cooperators, payoff_for_executors

# flexible incentives
class FlexibleIncentives(SocialDilemmaGame):
    def __init__(self, b, M, r, c, alpha, pi_e, pi_t, enhancement, N):
            super().__init__(b, M, r, c)
            self.alpha = alpha
            self.pi_e = pi_e
            self.pi_t = pi_t
            self.enhancement = enhancement
            self.N = N
    
    def payoff(self, j_c:int, j_e:int):
        p_d, _ = super().payoff(j_c + j_e)
        nominator = self.pi_t * self.enhancement * j_e
        payoff_for_defectors = p_d - (1 - self.alpha) * nominator / (self.N - j_e - j_c) * self.delta
        payoff_for_cooperators = p_d + self.alpha * nominator / (j_e + j_c) * self.delta - self.c
        payoff_for_executors = p_d + self.alpha * nominator / (j_e + j_c) * self.delta - self.c - self.pi_t
        return payoff_for_defectors, payoff_for_cooperators, payoff_for_executors

## 0.3) Evolutionary Dynamics

In [ ]:
class EvolutionaryDynamics:
    def __init__(self, game, Z:int, N:int):
        self.game = game
        self.Z = Z
        self.N = N
        self.beta = 5.0
        self.mu = mu # mutation

    # we can refactor this a bit?
    def average_payoffs(self, i_c: int, i_e: int):
        f_d = 0
        f_c = 0
        f_e = 0

        i_d = self.Z - i_c - i_e
        for j_c in range(0, self.N): # 0 - N - 1
            for j_e in range(0, self.N - (j_c + 1)): # 0 - N - 1 - j_c 
                factor_d = (sp.binom(i_c, j_c) * sp.binom(i_e, j_e) * sp.binom(self.Z - i_c - i_e - 1, self.N - 1 - j_c - j_e)) / sp.binom(self.Z - 1, self.N - 1)
                factor_c = (sp.binom(i_c - 1, j_c) * sp.binom(i_e, j_e) * sp.binom(self.Z - i_c - i_e, self.N - 1 - j_c - j_e)) / sp.binom(self.Z - 1, self.N - 1)
                factor_e = (sp.binom(i_c, j_c) * sp.binom(i_e - 1, j_e) * sp.binom(self.Z - i_c - i_e, self.N - 1 - j_c - j_e)) / sp.binom(self.Z - 1, self.N - 1)            
                p_d, _, _ = self.game.payoff(j_c, j_e)
                _, p_c, _ = self.game.payoff(j_c + 1, j_e)
                _, _, p_e = self.game.payoff(j_c, j_e + 1)
                f_d += factor_d * p_d
                f_c += factor_c * p_c
                f_e += factor_e * p_e
        return f_d, f_c, f_e

    def average_payoff_c(self, i_c,  i_e):
        f_c = 0

        for j_c in range(0, self.N): # 0 - N - 1
            for j_e in range(0, self.N - (j_c + 1)): # 0 - N - 1 - j_c 
                factor_c = (sp.binom(i_c - 1, j_c) * sp.binom(i_e, j_e) * sp.binom(self.Z - i_c - i_e, self.N - 1 - j_c - j_e)) / sp.binom(self.Z - 1, self.N - 1)
                _, p_c, _ = self.game.payoff(j_c + 1, j_e)
                f_c += factor_c * p_c
        return f_c

    def average_payoff_e(self, i_c, i_e):
        f_e = 0

        for j_c in range(0, self.N): # 0 - N - 1
            for j_e in range(0, self.N - (j_c + 1)): # 0 - N - 1 - j_c 
                factor_e = (sp.binom(i_c, j_c) * sp.binom(i_e - 1, j_e) * sp.binom(self.Z - i_c - i_e, self.N - 1 - j_c - j_e)) / sp.binom(self.Z - 1, self.N - 1)            
                _, _, p_e = self.game.payoff(j_c, j_e + 1)
                f_e += factor_e * p_e
        return f_e

    def average_payoff_d(self, i_c, i_e):
        f_d = 0

        for j_c in range(0, self.N): # 0 - N - 1
            for j_e in range(0, self.N - (j_c + 1)): # 0 - N - 1 - j_c 
                factor_d = (sp.binom(i_c, j_c) * sp.binom(i_e, j_e) * sp.binom(self.Z - i_c - i_e - 1, self.N - 1 - j_c - j_e)) / sp.binom(self.Z - 1, self.N - 1)          
                p_d, _, _ = self.game.payoff(j_c, j_e)
                f_d += factor_d * p_d

        return f_d

    # copied from assignment 1
    def fermifunc(self, first, second):
        '''
        The fermi function determines the probability that the first type
        imitates the second
        '''
        return 1./(1. + np.exp(-self.b*(first-second)))

    # this the function T L->R in paper
    def transition_probability(self, i_l, i_r, f_l, f_r):
        return (1 - self.mu) * (i_l / self.Z) * (i_r / (self.Z - 1)) * self.fermifunc(f_r, f_l) + self.mu * (i_l / (2 * self.Z))

    # validate whether T(ic, ie, id) = TL->R with the changed values as params?
    # TODO fix typos/bug in code
    def probability_increase_decrease(self, i_c, i_e, i_d):

        increase_c_decrease_e = self.transition_probability(i_c + 1, i_e - 1, self.average_payoff_c(i_c + 1, i_e - 1), self.average_payoff_e(i_c + 1, i_e - 1))
        decrease_c_increase_e = self.transition_probability(i_c - 1, i_e + 1, self.average_payoff_c(i_c - 1, i_e + 1), self.average_payoff_e(i_c - 1, i_e + 1))

        increase_c = increase_c_decrease_e + self.transition_probability(i_c + 1, i_d - 1, self.average_payoff_c(i_c + 1, i_d - 1), self.average_payoff_d(i_c + 1, i_d - 1))
        decrease_c = decrease_c_increase_e + self.transition_probability(i_c - 1, i_d + 1, self.average_payoff_c(i_c - 1, i_d + 1), self.average_payoff_d(i_c - 1, i_d + 1))

        increase_e = decrease_c_increase_e + self.transition_probability(i_e + 1, i_d - 1, self.average_payoff_e(i_c + 1, i_e - 1), self.average_payoff_d(i_c + 1, i_d - 1))
        decrease_e = increase_c_decrease_e + self.transition_probability(i_c - 1, i_d + 1, self.average_payoff_e(i_c - 1, i_e + 1), self.average_payoff_d(i_c - 1, i_d + 1))

        return increase_c, decrease_c, increase_e, decrease_e
    
# In each evolutionary time step, a randomly chosen individual with
# strategy L adopts the strategy R of another randomly selected individual from the population with proba-
# bility 1=½1 + ebðfR fLÞ

# 1) Reproduction Experiments

## 1.1) Fixed incentives under a local scheme

## 1.2) Flexible incentives under a local scheme

# 2) Extensions